# Waste image classification

Organic vs recyclable images: logistic regression on PCA features, an MLP, and a small CNN (with a short hyperparameter search). Runs on a GPU in Colab. The dataset is not in this repo (see `data/README.md`).

In [ ]:
import os

# Kaggle "Waste Classification data" (see data/README.md). Download the zip to data/waste-classification.zip,
# or in Colab upload it to /content/. It is extracted to a fast local folder.
source_zip = next(p for p in ['../data/waste-classification.zip', '/content/waste-classification.zip'] if os.path.exists(p))
target_dir = '/content/dta' if os.path.exists('/content') else '../data/waste-classification'

os.makedirs(target_dir, exist_ok=True)
print(f"Extracting {source_zip} to {target_dir} ...")
import zipfile
with zipfile.ZipFile(source_zip) as z:
    z.extractall(target_dir)

print("Done! Contents:", os.listdir(target_dir))


In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# 1. Identify Data Directory robustly
train_dir = None
for root, dirs, files in os.walk(target_dir):
    if 'TRAIN' in dirs:
        train_dir = os.path.join(root, 'TRAIN')
        break
    elif 'train' in dirs:
        train_dir = os.path.join(root, 'train')
        break

if train_dir is None:
    print("Could not find a 'TRAIN' directory. Please check the dataset structure.")
else:
    print(f"Loading images from: {train_dir}")

# 2. Pre-handle Images (Transformations)
transform = transforms.Compose([
    transforms.Resize((64, 64)), # Resizing to 64x64 for MLP efficiency
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 3. Load Dataset and Create Eval Set
full_dataset = datasets.ImageFolder(root=train_dir, transform=transform)

# Split into 80% train and 20% eval
train_size = int(0.8 * len(full_dataset))
eval_size = len(full_dataset) - train_size
train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size])

print(f"Detected Classes: {full_dataset.classes}")
print(f"Train set size: {len(train_dataset)}, Eval set size: {len(eval_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=2)

Loading images from: /content/dta/dataset/DATASET/TRAIN
Detected Classes: ['O', 'R']
Train set size: 18051, Eval set size: 4513


### Baseline: Logistic Regression with PCA
Before moving into Deep Learning, let's establish a baseline using Logistic Regression. Since image pixel data is very high-dimensional (64x64x3 = 12288), we apply **PCA (Principal Component Analysis)** first to extract the most important components and reduce noise.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
import time

# Helper to extract features into flat numpy arrays for Scikit-Learn
def get_flat_features(dataloader):
    features, labels_list = [], []
    for imgs, lbls in dataloader:
        features.append(imgs.view(imgs.size(0), -1).numpy())
        labels_list.append(lbls.numpy())
    return np.vstack(features), np.concatenate(labels_list)

print("Extracting flattening images for baseline...")
X_train, y_train = get_flat_features(train_loader)
X_eval, y_eval = get_flat_features(eval_loader)

print("Fitting PCA (Randomized solver, 128 components for speed)...")
# Standard PCA on 18k x 12k matrix freezes. We use randomized SVD for a massive speedup.
pca = PCA(n_components=128, svd_solver='randomized')
X_train_pca = pca.fit_transform(X_train)
X_eval_pca = pca.transform(X_eval)

print(f"Dimensions reduced from {X_train.shape[1]} to {X_train_pca.shape[1]}")
print(f"Variance Retained: {np.sum(pca.explained_variance_ratio_) * 100:.2f}%")

print("Training Logistic Regression Classifier...")
t_start = time.time()
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_pca, y_train)
print(f"Training took {time.time() - t_start:.2f}s")

logreg_eval_acc = accuracy_score(y_eval, logreg.predict(X_eval_pca))
print(f"Logistic Regression Eval Accuracy: {logreg_eval_acc * 100:.2f}%")

Extracting flattening images for baseline...


In [ ]:
# 4. Define the MLP Classifier with Dropout
class MLPClassifier(nn.Module):
    def __init__(self, input_size=64*64*3, hidden_size=512, num_classes=2, dropout_rate=0.5):
        super(MLPClassifier, self).__init__()
        self.flatten = nn.Flatten()

        # Fully Connected Layer 1
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate) # Dropout applied here

        # Fully Connected Layer 2
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate) # Dropout applied here

        # Output Layer
        self.fc3 = nn.Linear(hidden_size // 2, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        x = self.fc3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

model = MLPClassifier(num_classes=len(full_dataset.classes)).to(device)

In [ ]:
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim

# 5. Training Setup: Velocity, Lambda, Decreasing LR, Early Stopping
num_epochs = 20 # Increased slightly to allow early stopping to do its job
initial_lr = 0.01
momentum_velocity = 0.9 # Velocity
weight_decay_lambda = 1e-4 # Lambda (L2 Regularization)
patience = 3 # Early stopping patience

criterion = nn.CrossEntropyLoss()

# Optimizer with momentum and weight decay
optimizer = optim.SGD(model.parameters(),
                      lr=initial_lr,
                      momentum=momentum_velocity,
                      weight_decay=weight_decay_lambda)

# Decreasing Learning Rate: Multiplies LR by 0.5 every 3 epochs
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

print(f"Starting Training on device: {device}...")

best_eval_loss = float('inf')
patience_counter = 0
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    start_time = time.time()

    # --- TRAIN ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    # Update learning rate
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # --- EVALUATE ---
    model.eval()
    eval_loss, eval_correct, eval_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in eval_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            eval_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            eval_total += labels.size(0)
            eval_correct += (predicted == labels).sum().item()

    epoch_duration = time.time() - start_time
    avg_train_loss = train_loss / train_total
    avg_eval_loss = eval_loss / eval_total

    print(f"Epoch {epoch+1:02d}/{num_epochs} [{epoch_duration:.1f}s] | LR: {current_lr:.5f}")
    print(f"  Train -> Loss: {avg_train_loss:.4f}, Acc: {100*train_correct/train_total:.2f}%")
    print(f"  Eval  -> Loss: {avg_eval_loss:.4f}, Acc: {100*eval_correct/eval_total:.2f}%")

    # --- EARLY STOPPING ---
    if avg_eval_loss < best_eval_loss:
        best_eval_loss = avg_eval_loss
        patience_counter = 0
        best_model_weights = copy.deepcopy(model.state_dict())
        print(f"  [*] New best eval loss: {best_eval_loss:.4f}! Model saved.")
    else:
        patience_counter += 1
        print(f"  [!] No improvement in eval loss. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print("Early stopping triggered. Halting training.")
            break

# Load the best model weights back into the model before finishing
model.load_state_dict(best_model_weights)
print("Training Complete! Loaded best model weights.")

In [ ]:
import os
from torchvision import datasets
from torch.utils.data import DataLoader
import torch

# 6. Locate and Evaluate on Test Data
test_dir = None
for root, dirs, files in os.walk(target_dir):
    if 'TEST' in dirs:
        test_dir = os.path.join(root, 'TEST')
        break
    elif 'test' in dirs:
        test_dir = os.path.join(root, 'test')
        break

if test_dir is None:
    print("Could not find a 'TEST' directory. Please check the dataset structure.")
else:
    print(f"Found test data at: {test_dir}")

    # Re-use the transform defined earlier
    test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

    print(f"Test classes detected: {test_dataset.classes}")
    print(f"Test set size: {len(test_dataset)}")

    print("\nRunning Test Evaluation...")
    model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    if test_total > 0:
        avg_test_loss = test_loss / test_total
        test_accuracy = 100 * test_correct / test_total
        mlp_test_acc = test_accuracy
        print(f"Test Results -> Loss: {avg_test_loss:.4f} | Accuracy: {test_accuracy:.2f}%")
    else:
        print("No test images found to evaluate.")


In [ ]:
# 7. Define a Convolutional Neural Network (CNN)
class CNNClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super(CNNClassifier, self).__init__()

        # Convolutional Block 1: 64x64x3 -> 32x32x16
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Convolutional Block 2: 32x32x16 -> 16x16x32
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Convolutional Block 3: 16x16x32 -> 8x8x64
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Layers
        # After 3 max pools of size 2, our 64x64 image is now 8x8.
        # 64 channels * 8 width * 8 height = 4096 features
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.relu_fc = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.flatten(x)
        x = self.dropout(self.relu_fc(self.fc1(x)))
        x = self.fc2(x)
        return x

# Initialize the CNN
cnn_model = CNNClassifier(num_classes=len(test_dataset.classes)).to(device)
print(f"CNN Model created and moved to {device}")

In [ ]:
# 8. Train the CNN
# We will use Adam optimizer this time, which often converges faster for CNNs
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=0.001, weight_decay=1e-4)
cnn_scheduler = optim.lr_scheduler.StepLR(cnn_optimizer, step_size=4, gamma=0.5)
cnn_criterion = nn.CrossEntropyLoss()

num_epochs_cnn = 15
best_cnn_eval_loss = float('inf')
cnn_patience = 3
cnn_patience_counter = 0
best_cnn_weights = copy.deepcopy(cnn_model.state_dict())

print("Starting CNN Training...")

for epoch in range(num_epochs_cnn):
    start_time = time.time()

    # --- TRAIN ---
    cnn_model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        cnn_optimizer.zero_grad()
        outputs = cnn_model(images)
        loss = cnn_criterion(outputs, labels)
        loss.backward()
        cnn_optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    cnn_scheduler.step()
    current_lr = cnn_scheduler.get_last_lr()[0]

    # --- EVALUATE ---
    cnn_model.eval()
    eval_loss, eval_correct, eval_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in eval_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cnn_model(images)
            loss = cnn_criterion(outputs, labels)

            eval_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            eval_total += labels.size(0)
            eval_correct += (predicted == labels).sum().item()

    epoch_duration = time.time() - start_time
    avg_train_loss = train_loss / train_total
    avg_eval_loss = eval_loss / eval_total

    print(f"Epoch {epoch+1:02d}/{num_epochs_cnn} [{epoch_duration:.1f}s] | LR: {current_lr:.5f}")
    print(f"  Train -> Loss: {avg_train_loss:.4f}, Acc: {100*train_correct/train_total:.2f}%")
    print(f"  Eval  -> Loss: {avg_eval_loss:.4f}, Acc: {100*eval_correct/eval_total:.2f}%")

    # --- EARLY STOPPING ---
    if avg_eval_loss < best_cnn_eval_loss:
        best_cnn_eval_loss = avg_eval_loss
        cnn_patience_counter = 0
        best_cnn_weights = copy.deepcopy(cnn_model.state_dict())
    else:
        cnn_patience_counter += 1
        if cnn_patience_counter >= cnn_patience:
            print("  [!] Early stopping triggered.")
            break

cnn_model.load_state_dict(best_cnn_weights)
print("CNN Training Complete!")

In [ ]:
# 9. Evaluate CNN on Test Data
print("\nRunning CNN Test Evaluation...")
cnn_model.eval()
test_loss, test_correct, test_total = 0.0, 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cnn_model(images)
        loss = cnn_criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

if test_total > 0:
    avg_test_loss = test_loss / test_total
    test_accuracy = 100 * test_correct / test_total
    print(f"CNN Test Results -> Loss: {avg_test_loss:.4f} | Accuracy: {test_accuracy:.2f}%")
else:
    print("No test images found to evaluate.")

### Re-train CNN (More Epochs) & Visual Comparison
We will re-train the CNN for more epochs (up to 40) while relying on Early Stopping (patience=5) to halt training when it stops improving. After that, we evaluate all models on the `TEST` set and plot the results.

*(Note on PCA for Deep Learning: While PCA helps linear models and sometimes basic MLPs, it destroys the 2D spatial pixel structures that CNNs explicitly rely on. Therefore, we do not apply PCA to CNN inputs.)*

In [ ]:
print("Re-training CNN with more epochs...")
# Re-initialize model to start fresh
cnn_model_v2 = CNNClassifier(num_classes=len(test_dataset.classes)).to(device)
cnn_optimizer_v2 = optim.Adam(cnn_model_v2.parameters(), lr=0.001, weight_decay=1e-4)
cnn_scheduler_v2 = optim.lr_scheduler.StepLR(cnn_optimizer_v2, step_size=5, gamma=0.5)

num_epochs_long = 40
patience_long = 5
best_loss_v2 = float('inf')
patience_cnt = 0
best_w_v2 = copy.deepcopy(cnn_model_v2.state_dict())

for epoch in range(num_epochs_long):
    # Train
    cnn_model_v2.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        cnn_optimizer_v2.zero_grad()
        loss = cnn_criterion(cnn_model_v2(images), labels)
        loss.backward()
        cnn_optimizer_v2.step()
    cnn_scheduler_v2.step()

    # Eval
    cnn_model_v2.eval()
    eval_loss = 0.0
    with torch.no_grad():
        for images, labels in eval_loader:
            images, labels = images.to(device), labels.to(device)
            loss = cnn_criterion(cnn_model_v2(images), labels)
            eval_loss += loss.item() * images.size(0)
    avg_eval_loss = eval_loss / len(eval_dataset)

    print(f"Epoch {epoch+1:02d}/{num_epochs_long} | Eval Loss: {avg_eval_loss:.4f}")

    # Early stopping
    if avg_eval_loss < best_loss_v2:
        best_loss_v2 = avg_eval_loss
        patience_cnt = 0
        best_w_v2 = copy.deepcopy(cnn_model_v2.state_dict())
    else:
        patience_cnt += 1
        if patience_cnt >= patience_long:
            print(f"Early stopping at epoch {epoch+1}")
            break

cnn_model_v2.load_state_dict(best_w_v2)
print("CNN Re-training Complete!")

In [ ]:
import matplotlib.pyplot as plt

# 1. Test Logistic Regression
X_test, y_test = get_flat_features(test_loader)
X_test_pca = pca.transform(X_test)
log_test_acc = accuracy_score(y_test, logreg.predict(X_test_pca)) * 100
print(f"Logistic Regression Test Accuracy: {log_test_acc:.2f}%")

# 2. Test Re-trained CNN
cnn_model_v2.eval()
test_correct_cnn = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = torch.max(cnn_model_v2(images).data, 1)
        test_correct_cnn += (predicted == labels).sum().item()
cnn_test_acc = 100 * test_correct_cnn / len(test_dataset)
print(f"Improved CNN Test Accuracy: {cnn_test_acc:.2f}%")

# 3. MLP test accuracy comes from the MLP test evaluation above (mlp_test_acc)

# 4. Plot Comparison
models = ['Logistic Regression\n(+ PCA)', 'MLP\n(Raw Pixels)', 'CNN\n(More Epochs)']
accuracies = [log_test_acc, mlp_test_acc, cnn_test_acc]

plt.figure(figsize=(9, 6))
bars = plt.bar(models, accuracies, color=['#ff9999','#66b3ff','#99ff99'])
plt.ylim(0, 100)
plt.ylabel('Test Accuracy (%)')
plt.title('Model Performance Comparison')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1.5, f'{yval:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import torch

# 1. Gather Ground Truth and Predictions for all models
y_true = y_test # Ground truth from the test set

# --- Logistic Regression Predictions ---
y_pred_lr = logreg.predict(X_test_pca)

# --- MLP Predictions ---
model.eval()
y_pred_mlp_list = []
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        y_pred_mlp_list.append(predicted.cpu().numpy())
y_pred_mlp = np.concatenate(y_pred_mlp_list)

# --- CNN (v2) Predictions ---
cnn_model_v2.eval()
y_pred_cnn_list = []
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = cnn_model_v2(images)
        _, predicted = torch.max(outputs, 1)
        y_pred_cnn_list.append(predicted.cpu().numpy())
y_pred_cnn = np.concatenate(y_pred_cnn_list)

# 2. Compute Metrics
metrics = {}
for name, y_pred in [('Logistic Regression', y_pred_lr), ('MLP', y_pred_mlp), ('CNN (v2)', y_pred_cnn)]:
    acc = accuracy_score(y_true, y_pred)
    # Compute macro-averaged precision, recall, and f1-score
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    metrics[name] = {
        'Accuracy': acc * 100,
        'Precision': prec * 100,
        'Recall': rec * 100,
        'F1-Score': f1 * 100
    }

# 3. Print the metrics table
print(f"{'Model':<25} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
print("-" * 71)
for model_name, m in metrics.items():
    print(f"{model_name:<25} | {m['Accuracy']:0.2f}%     | {m['Precision']:0.2f}%     | {m['Recall']:0.2f}%     | {m['F1-Score']:0.2f}%")

# 4. Plot Grouped Bar Chart
labels = list(metrics.keys())
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(labels))  # label locations
width = 0.2  # width of the bars

fig, ax = plt.subplots(figsize=(11, 6.5))

# Draw bars for each metric
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']
for i, metric in enumerate(metric_names):
    values = [metrics[model_name][metric] for model_name in labels]
    rects = ax.bar(x + i*width - 1.5*width, values, width, label=metric, color=colors[i])
    # Add values on top of bars
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Detailed Model Evaluation Comparison on Test Set', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 110)
ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='lightgray')
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import itertools
import copy

# Define hyperparameter search space
learning_rates = [0.005, 0.001, 0.0005]
weight_decays = [1e-4, 1e-3]
best_val_acc = 0.0
best_params = {}
best_cnn_model_state = None

# Grid Search
for lr, wd in itertools.product(learning_rates, weight_decays):
    print(f"\nEvaluating: LR={lr}, Weight Decay={wd}...")

    # Initialize a fresh model instance
    trial_model = CNNClassifier(num_classes=len(test_dataset.classes)).to(device)
    optimizer = optim.Adam(trial_model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss()

    # Quick training: 5 epochs per configuration to quickly find the best setting
    for epoch in range(5):
        trial_model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(trial_model(images), labels)
            loss.backward()
            optimizer.step()

    # Evaluate on eval set
    trial_model.eval()
    eval_correct = 0
    eval_total = 0
    with torch.no_grad():
        for images, labels in eval_loader:
            images, labels = images.to(device), labels.to(device)
            _, predicted = torch.max(trial_model(images).data, 1)
            eval_total += labels.size(0)
            eval_correct += (predicted == labels).sum().item()

    val_acc = 100 * eval_correct / eval_total
    print(f"-> Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_params = {'learning_rate': lr, 'weight_decay': wd}
        best_cnn_model_state = copy.deepcopy(trial_model.state_dict())

print(f"\nBest Hyperparameters Found: {best_params} with Validation Accuracy: {best_val_acc:.2f}%")

In [ ]:
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np

# 1. Initialize CNN with Best Hyperparameters
final_optimal_cnn = CNNClassifier(num_classes=len(test_dataset.classes)).to(device)
opt_optimizer = optim.Adam(final_optimal_cnn.parameters(), lr=0.001, weight_decay=0.001)
opt_criterion = nn.CrossEntropyLoss()

num_epochs_opt = 30
patience_opt = 5
best_val_loss_opt = float('inf')
patience_counter_opt = 0
best_opt_weights = copy.deepcopy(final_optimal_cnn.state_dict())

# Lists to keep track of losses for plotting
train_losses = []
val_losses = []

print("Retraining CNN on CUDA with Best Hyperparameters (LR=0.001, WD=0.001)...")

for epoch in range(num_epochs_opt):
    # --- TRAIN ---
    final_optimal_cnn.train()
    running_train_loss = 0.0
    total_train_images = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        opt_optimizer.zero_grad()
        outputs = final_optimal_cnn(images)
        loss = opt_criterion(outputs, labels)
        loss.backward()
        opt_optimizer.step()

        running_train_loss += loss.item() * images.size(0)
        total_train_images += images.size(0)

    epoch_train_loss = running_train_loss / total_train_images
    train_losses.append(epoch_train_loss)

    # --- VALIDATE ---
    final_optimal_cnn.eval()
    running_val_loss = 0.0
    total_val_images = 0
    with torch.no_grad():
        for images, labels in eval_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = final_optimal_cnn(images)
            loss = opt_criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)
            total_val_images += images.size(0)

    epoch_val_loss = running_val_loss / total_val_images
    val_losses.append(epoch_val_loss)

    print(f"Epoch {epoch+1:02d}/{num_epochs_opt} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

    # --- EARLY STOPPING ---
    if epoch_val_loss < best_val_loss_opt:
        best_val_loss_opt = epoch_val_loss
        patience_counter_opt = 0
        best_opt_weights = copy.deepcopy(final_optimal_cnn.state_dict())
    else:
        patience_counter_opt += 1
        if patience_counter_opt >= patience_opt:
            print(f"Early stopping triggered at epoch {epoch+1}.")
            break

# Load best weights
final_optimal_cnn.load_state_dict(best_opt_weights)
print("Retraining complete! Best weights loaded.")

In [ ]:
# 2. Plot Training and Validation Loss Curves
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss', color='#1f77b4', marker='o')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss', color='#ff7f0e', marker='s')
plt.title('CNN Loss Curves (Optimal Hyperparameters)', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(range(1, len(train_losses) + 1))
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Evaluate the Retrained Optimized CNN model on Test Dataset
final_optimal_cnn.eval()
y_pred_opt_cnn_list = []
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = final_optimal_cnn(images)
        _, predicted = torch.max(outputs, 1)
        y_pred_opt_cnn_list.append(predicted.cpu().numpy())
y_pred_opt_cnn = np.concatenate(y_pred_opt_cnn_list)

# Compute test metrics for the final optimized CNN
opt_acc = accuracy_score(y_true, y_pred_opt_cnn)
opt_prec, opt_rec, opt_f1, _ = precision_recall_fscore_support(y_true, y_pred_opt_cnn, average='macro')

# Add the new optimized CNN metrics to our comparison pool
metrics['Optimized CNN (v3)'] = {
    'Accuracy': opt_acc * 100,
    'Precision': opt_prec * 100,
    'Recall': opt_rec * 100,
    'F1-Score': opt_f1 * 100
}

# 4. Print Updated Metrics Table
print(f"{'Model':<25} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
print("-" * 71)
for model_name, m in metrics.items():
    print(f"{model_name:<25} | {m['Accuracy']:0.2f}%     | {m['Precision']:0.2f}%     | {m['Recall']:0.2f}%     | {m['F1-Score']:0.2f}%")

# 5. Plot Grouped Bar Chart of All Models
labels = list(metrics.keys())
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(labels))
width = 0.18

fig, ax = plt.subplots(figsize=(13, 7))
c_palette = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

for i, metric in enumerate(metric_names):
    values = [metrics[m_name][metric] for model_name, m_name in enumerate(labels)]
    rects = ax.bar(x + i*width - 1.5*width, values, width, label=metric, color=c_palette[i])
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Detailed Model Evaluation Comparison on Test Set (with Optimized CNN)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 115)
ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='lightgray')
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()